# Week 6: Integrated Data Science Project
**Prepared by:** Tushar Sharma

Research question: Can transaction attributes help explain and predict recorded transaction spend, and what patterns appear across categories and time? This notebook integrates data cleaning, EDA, statistical modeling, machine learning, and visualization.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(Path("../data/retail_store_sales_raw.csv"))
for col in ["Price Per Unit", "Quantity", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
df["Discount Applied"] = df["Discount Applied"].astype("boolean")
for col in ["Transaction ID", "Customer ID", "Category", "Item", "Payment Method", "Location"]:
    df[col] = df[col].astype("string").str.strip()
print("Shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
display(df.head())

## 1. Data quality and descriptive statistics

In [ ]:
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}))
display(df[["Price Per Unit", "Quantity", "Total Spent"]].describe().T)

## 2. Exploratory analysis and visualization

In [ ]:
category_spend = df.groupby("Category")["Total Spent"].sum(min_count=1).sort_values(ascending=False)
display(category_spend.to_frame("Recorded spend"))
category_spend.sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Recorded Spend by Category"); plt.xlabel("Total Spent"); plt.ylabel("Category")
plt.tight_layout(); plt.show()

dated = df.dropna(subset=["Transaction Date", "Total Spent"]).copy()
dated["Month"] = dated["Transaction Date"].dt.to_period("M").astype(str)
monthly = dated.groupby("Month")["Total Spent"].sum()
monthly.plot(figsize=(10, 4.5))
plt.title("Monthly Recorded Spend"); plt.xlabel("Month"); plt.ylabel("Total Spent")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 3. Statistical regression model
Fit an OLS regression on rows with complete price, quantity, and total spend. The relationship is partly structural because total spend is linked to price and quantity.

In [ ]:
numeric = df[["Price Per Unit", "Quantity", "Total Spent"]].dropna()
X_stat = sm.add_constant(numeric[["Price Per Unit", "Quantity"]])
stat_model = sm.OLS(numeric["Total Spent"], X_stat).fit()
print(stat_model.summary())

## 4. Machine learning model and tuning

In [ ]:
target = "Total Spent"
features = ["Price Per Unit", "Quantity", "Category", "Payment Method", "Location", "Discount Applied"]
model_df = df[features + [target]].dropna(subset=[target]).copy()
X, y = model_df[features], model_df[target]
num_features = ["Price Per Unit", "Quantity"]
cat_features = ["Category", "Payment Method", "Location", "Discount Applied"]
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
baseline = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

rf = Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(random_state=42, n_jobs=-1))])
grid = {"model__n_estimators": [100, 200], "model__max_depth": [None, 10], "model__min_samples_leaf": [1, 3]}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(rf, grid, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1)
search.fit(X_train, y_train)
rf_pred = search.best_estimator_.predict(X_test)
print("Best parameters:", search.best_params_)

In [ ]:
def metrics(name, actual, predicted):
    return {"Model": name, "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)), "R2": r2_score(actual, predicted)}
results = pd.DataFrame([metrics("Linear Regression", y_test, base_pred),
                        metrics("Tuned Random Forest", y_test, rf_pred)])
display(results.round(3))

## 5. Conclusion and limitations
This integrated workflow combines quality checks, descriptive analysis, visualizations, OLS regression, and a tuned Random Forest. Interpret all metrics in context. Since Total Spent is closely related to Price Per Unit × Quantity, performance may reflect this built-in formula. Missing values and the structured nature of the dataset limit external generalization. A future improvement would be to test on a separate, later time period or a different dataset.